In [4]:
#2021.10.13. WED
#Team_SmokeFree

## Data Preprocessing, JUSO -> X, Y, GU, HGDONG_CODE, HGDONG
#00. 패키지 호출하기.
import warnings
import pandas as pd 
import requests
from urllib.parse import quote

#00-1. warning message ignore 
warnings.filterwarnings(action='ignore')

#01. kakao_api_key.txt 파일 부르기. 
with open("A:/Python_Project/Team_SmokeFree/key/kakao_api_key.txt",mode="r") as key_file :
    kakao_api_key = key_file.read()

#02. 법정동을 행정동으로 변환하기. 
#(1) 데이터셋 불러오기. 
SS_data = pd.read_excel('../data/01_Feature_Data/서울특별시_담배소매업_pp.xlsx')
dong_data = pd.read_excel('../data/서울특별시_행정동_법정동_코드.xlsx')

#(2) 처리 결과 확인하기. 
SS_data

,자치구,행정동_코드,행정동,사업장명,소재지_지번_주소,소재지_도로명_주소,위도,경도
0,NaN,NaN,NaN,지에스25 금천다온,서울특별시 금천구 시흥동 923번지,서울특별시 금천구 독산로6가길 18 1층 일부호 (시흥동),NaN,NaN
1,NaN,NaN,NaN,씨유 선인상가점,NaN,서울특별시 용산구 새창로 181 21동 2018호 (한강로2가 선인상가),NaN,NaN
2,NaN,NaN,NaN,지에스(GS)25영휘원점,서울특별시 동대문구 청량리동 199번지 121호,서울특별시 동대문구 홍릉로 78-1 (청량리동),NaN,NaN
3,NaN,NaN,NaN,이마트24동작골든포레점,서울특별시 동작구 사당동 181번지 사당 롯데캐슬 골든포레,서울특별시 동작구 사당로 90 상가1동 105호 (사당동 사당 롯데캐슬 골든포레),NaN,NaN
4,NaN,NaN,NaN,지에스25 중곡희망점,NaN,서울특별시 광진구 동일로68길 49 1층 (중곡동 월드빌),NaN,NaN
...,...,...,...,...,...,...,...,...
20496,NaN,NaN,NaN,씨유 강북오패산로점,서울특별시 강북구 미아동 245-32,서울특별시 강북구 오패산로 271 1층 (미아동),NaN,NaN
20497,NaN,NaN,NaN,GS25 신월초교점,서울특별시 강서구 화곡동 1081-7,서울특별시 강서구 화곡로24길 14(화곡동),NaN,NaN
20498,NaN,NaN,NaN,럭키스토어,서울특별시 은평구 역촌동 2-38 SM 벨리체,서울특별시 은평구 서오릉로 107 1층 104호 (역촌동 SM 벨리체),NaN,NaN
20499,NaN,NaN,NaN,온오프,서울특별시 동대문구 장안동 414-8,서울특별시 동대문구 천호대로 365(장안동),NaN,NaN


In [5]:
dong_data

,시도명,시군구명,행정동코드,행정동,법정동코드,법정동
0,서울특별시,종로구,1111000000,NaN,1111000000,종로구
1,서울특별시,종로구,1111051500,청운효자동,1111010100,청운동
2,서울특별시,종로구,1111051500,청운효자동,1111010200,신교동
3,서울특별시,종로구,1111051500,청운효자동,1111010300,궁정동
4,서울특별시,종로구,1111051500,청운효자동,1111010400,효자동
...,...,...,...,...,...,...
763,서울특별시,강동구,1174065000,성내제2동,1174010800,성내동
764,서울특별시,강동구,1174066000,성내제3동,1174010800,성내동
765,서울특별시,강동구,1174068500,길동,1174010500,길동
766,서울특별시,강동구,1174069000,둔촌제1동,1174010600,둔촌동


In [6]:
#03. API 이용하여 정보 추출하기. 
#(1) 주소지를 기반으로 행정동 코드, 경도, 위도 추출하여 데이터프레임에 기입하기. 
local_url = "https://dapi.kakao.com/v2/local/search/address.json"
for num in range(len(SS_data))                                                                      :
    try                                                                                             :
        keyword = SS_data['소재지_도로명_주소'][num]   
        url = f'{local_url}?query={quote(keyword)}' 
    except Exception                                                                                :
        keyword = SS_data['소재지_지번_주소'][num]
        url = f'{local_url}?query={quote(keyword)}'

    try                                                                                             : 
        REQ_RESULT = requests.get(url, headers={"Authorization":f"KakaoAK {kakao_api_key}"}).json()
        SS_data['자치구'][num] = REQ_RESULT['documents'][0]['address']['region_2depth_name']
        SS_data['행정동_코드'][num] = REQ_RESULT["documents"][0]['address']['h_code']
        SS_data['경도'][num] = REQ_RESULT["documents"][0]["x"]
        SS_data['위도'][num] = REQ_RESULT["documents"][0]["y"]
        #print(f'DONE ! index_num = {num}')
    except Exception                                                                                :
        try                                                                                         :
            keyword = SS_data['소재지_지번_주소'][num]
            url=f'{local_url}?query={quote(keyword)}'
            REQ_RESULT = requests.get(url, headers={"Authorization":f"KakaoAK {kakao_api_key}"}).json()
            SS_data['자치구'][num] = REQ_RESULT['documents'][0]['address']['region_2depth_name']
            SS_data['행정동_코드'][num] = REQ_RESULT["documents"][0]['address']['h_code']
            SS_data['경도'][num] = REQ_RESULT["documents"][0]["x"]
            SS_data['위도'][num] = REQ_RESULT["documents"][0]["y"]
            #print(f'DONE ! index_num = {num}')
        except Exception                                                                            :
            SS_data['자치구'][num] = 'ERROR'
            SS_data['행정동_코드'][num] = 0
            SS_data['경도'][num] = 0
            SS_data['위도'][num] = 0
            #print(f'ERROR ! index_num = {num}')

#(2) 처리 결과 확인하기. 
SS_data

,자치구,행정동_코드,행정동,사업장명,소재지_지번_주소,소재지_도로명_주소,위도,경도
0,금천구,1154571000.0,NaN,지에스25 금천다온,서울특별시 금천구 시흥동 923번지,서울특별시 금천구 독산로6가길 18 1층 일부호 (시흥동),37.448859,126.905963
1,용산구,1117062500.0,NaN,씨유 선인상가점,NaN,서울특별시 용산구 새창로 181 21동 2018호 (한강로2가 선인상가),37.532717,126.965178
2,동대문구,1123070500.0,NaN,지에스(GS)25영휘원점,서울특별시 동대문구 청량리동 199번지 121호,서울특별시 동대문구 홍릉로 78-1 (청량리동),37.587201,127.043224
3,동작구,1159065100.0,NaN,이마트24동작골든포레점,서울특별시 동작구 사당동 181번지 사당 롯데캐슬 골든포레,서울특별시 동작구 사당로 90 상가1동 105호 (사당동 사당 롯데캐슬 골든포레),37.492336,126.962448
4,광진구,1121574000.0,NaN,지에스25 중곡희망점,NaN,서울특별시 광진구 동일로68길 49 1층 (중곡동 월드빌),37.562168,127.078546
...,...,...,...,...,...,...,...,...
20496,강북구,1130553500,NaN,씨유 강북오패산로점,서울특별시 강북구 미아동 245-32,서울특별시 강북구 오패산로 271 1층 (미아동),37.625541,127.029678
20497,강서구,1150054000,NaN,GS25 신월초교점,서울특별시 강서구 화곡동 1081-7,서울특별시 강서구 화곡로24길 14(화곡동),37.540314,126.838198
20498,은평구,1138062500,NaN,럭키스토어,서울특별시 은평구 역촌동 2-38 SM 벨리체,서울특별시 은평구 서오릉로 107 1층 104호 (역촌동 SM 벨리체),37.609286,126.919912
20499,동대문구,1123065000,NaN,온오프,서울특별시 동대문구 장안동 414-8,서울특별시 동대문구 천호대로 365(장안동),37.562558,127.059736


In [7]:
#04. 행정동 코드를 기반으로 행정동 값 매칭하기. 
#(1) FOR-LOOP 사용하기.
for i in range(len(SS_data))                                       :
    for k in range(len(dong_data))                                 :
        if SS_data['행정동_코드'][i] == dong_data['행정동코드'][k] : 
            SS_data['행정동'][i] = dong_data['행정동'][k]
            break

#(2) 처리 결과 확인하기. 
SS_data

,자치구,행정동_코드,행정동,사업장명,소재지_지번_주소,소재지_도로명_주소,위도,경도
0,금천구,1154571000.0,시흥제5동,지에스25 금천다온,서울특별시 금천구 시흥동 923번지,서울특별시 금천구 독산로6가길 18 1층 일부호 (시흥동),37.448859,126.905963
1,용산구,1117062500.0,한강로동,씨유 선인상가점,NaN,서울특별시 용산구 새창로 181 21동 2018호 (한강로2가 선인상가),37.532717,126.965178
2,동대문구,1123070500.0,청량리동,지에스(GS)25영휘원점,서울특별시 동대문구 청량리동 199번지 121호,서울특별시 동대문구 홍릉로 78-1 (청량리동),37.587201,127.043224
3,동작구,1159065100.0,사당제5동,이마트24동작골든포레점,서울특별시 동작구 사당동 181번지 사당 롯데캐슬 골든포레,서울특별시 동작구 사당로 90 상가1동 105호 (사당동 사당 롯데캐슬 골든포레),37.492336,126.962448
4,광진구,1121574000.0,중곡제1동,지에스25 중곡희망점,NaN,서울특별시 광진구 동일로68길 49 1층 (중곡동 월드빌),37.562168,127.078546
...,...,...,...,...,...,...,...,...
20496,강북구,1130553500,NaN,씨유 강북오패산로점,서울특별시 강북구 미아동 245-32,서울특별시 강북구 오패산로 271 1층 (미아동),37.625541,127.029678
20497,강서구,1150054000,NaN,GS25 신월초교점,서울특별시 강서구 화곡동 1081-7,서울특별시 강서구 화곡로24길 14(화곡동),37.540314,126.838198
20498,은평구,1138062500,NaN,럭키스토어,서울특별시 은평구 역촌동 2-38 SM 벨리체,서울특별시 은평구 서오릉로 107 1층 104호 (역촌동 SM 벨리체),37.609286,126.919912
20499,동대문구,1123065000,NaN,온오프,서울특별시 동대문구 장안동 414-8,서울특별시 동대문구 천호대로 365(장안동),37.562558,127.059736


In [9]:
#(3) 처리 결과 저장하기.
SS_data.to_excel('temp_file.xlsx', index=True)